# Pinecone을 사용한 Retrieval-Augmented Generation

이 notebook은 retrieval-augmented generation (RAG)이라는 기법으로 Claude를 Pinecone vector database의 데이터에 연결하는 방법을 보여줍니다. 다음 단계를 다룹니다:

1. Voyage AI의 embedding 모델로 dataset을 embedding
2. Embedding을 Pinecone index에 업로드
3. Vector database에서 정보 retrieval
4. Claude를 사용하여 database의 정보로 질문에 답하기

## 설정
먼저 필요한 라이브러리를 설치하고 이 notebook에서 사용할 API key를 설정하겠습니다. [Claude API key](https://docs.claude.com/claude/reference/getting-started-with-the-api), 무료 [Pinecone API key](https://docs.pinecone.io/docs/quickstart), 무료 [Voyage AI API key](https://docs.voyageai.com/install/)가 필요합니다. 

In [ ]:
%pip install anthropic datasets pinecone-client voyageai

In [ ]:
# 여기에 API key 입력
ANTHROPIC_API_KEY = "<YOUR_ANTHROPIC_API_KEY>"
PINECONE_API_KEY = "<YOUR_PINECONE_API_KEY>"
VOYAGE_API_KEY = "<YOUR_VOYAGE_API_KEY>"

## Dataset 다운로드
이제 Amazon 제품 설명이 1만 개 이상 포함된 Amazon 제품 dataset을 다운로드하고 DataFrame에 로드하겠습니다.

In [ ]:
import pandas as pd

# JSONL 파일 다운로드
!wget  https://www-cdn.anthropic.com/48affa556a5af1de657d426bcc1506cdf7e2f68e/amazon-products.jsonl

data = []
with open("amazon-products.jsonl") as file:
    for line in file:
        try:
            data.append(eval(line))  # noqa: S307
        except (SyntaxError, ValueError):
            # Dataset의 잘못된 형식의 줄 건너뛰기
            pass

df = pd.DataFrame(data)
display(df.head())
len(df)

## Vector Database

Vector database를 만들려면 먼저 Pinecone의 무료 API key가 필요합니다. Key가 준비되면 다음과 같이 database를 초기화할 수 있습니다:

In [ ]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

다음으로 index를 배포할 cloud provider와 region을 정의할 수 있는 index specification을 설정합니다. 사용 가능한 모든 provider와 region 목록은 [여기](https://www.pinecone.io/docs/data-types/metadata/)에서 확인할 수 있습니다.


In [ ]:
from pinecone import ServerlessSpec

spec = ServerlessSpec(cloud="aws", region="us-west-2")

그런 다음 index를 초기화합니다. Embedding 생성에는 Voyage의 "voyage-2" 모델을 사용하므로 dimension을 1024로 설정합니다.

In [ ]:
import time

index_name = "amazon-products"
existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

# Index가 이미 있는지 확인(처음이라면 없어야 함)
if index_name not in existing_indexes:
    # 없으면 index 생성
    pc.create_index(
        index_name,
        dimension=1024,  # voyage-2 embedding의 dimension
        metric="dotproduct",
        spec=spec,
    )
    # Index 초기화 대기
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

# Index에 연결
index = pc.Index(index_name)
time.sleep(1)
# Index 통계 보기
index.describe_index_stats()

아직 vector를 추가하지 않았으므로 새 Pinecone index의 total_vector_count가 0인 것을 확인할 수 있어야 합니다.

## Embedding
Voyage의 embedding을 시작하려면 [여기](https://www.voyageai.com)에서 API key를 발급받으세요.

이제 Voyage client를 설정하고 `embed` method를 사용하여 embedding을 생성하는 방법을 보여주겠습니다. Voyage embedding을 Claude와 함께 사용하는 자세한 방법은 [이 notebook](https://github.com/anthropics/anthropic-cookbook/blob/main/third_party/VoyageAI/how_to_create_embeddings.md)을 참조하세요.

In [ ]:
import voyageai

vo = voyageai.Client(api_key=VOYAGE_API_KEY)

texts = ["Sample text 1", "Sample text 2"]

result = vo.embed(texts, model="voyage-2", input_type="document")
print(result.embeddings[0])
print(result.embeddings[1])

## Pinecone index에 데이터 업로드

Embedding 모델이 설정되었으므로 이제 제품 설명을 embedding하고 embedding을 Pinecone index에 업로드할 수 있습니다.

In [ ]:
from time import sleep

from tqdm.auto import tqdm

descriptions = df["text"].tolist()
batch_size = 100  # 한 번에 생성하고 삽입할 embedding 수

for i in tqdm(range(0, len(descriptions), batch_size)):
    # Batch 끝 찾기
    i_end = min(len(descriptions), i + batch_size)
    descriptions_batch = descriptions[i:i_end]
    # Embedding 생성(RateLimitError 방지를 위해 try-except 추가. Voyage는 현재 분당 300회 요청 허용.)
    done = False
    while not done:
        try:
            res = vo.embed(descriptions_batch, model="voyage-2", input_type="document")
            done = True
        except Exception:
            sleep(5)

    embeds = [record for record in res.embeddings]
    # 각 텍스트의 unique ID 생성
    ids_batch = [f"description_{idx}" for idx in range(i, i_end)]

    # 각 텍스트의 metadata dictionary 생성
    metadata_batch = [{"description": description} for description in descriptions_batch]

    to_upsert = list(zip(ids_batch, embeds, metadata_batch, strict=False))

    # Pinecone에 upsert
    index.upsert(vectors=to_upsert)

## Query 실행

Index에 데이터가 채워졌으므로 이제 query를 실행하여 결과를 얻을 수 있습니다. 자연어 질문을 embedding하고 index에 query하여 의미적으로 유사한 제품 설명을 반환할 수 있습니다.

In [49]:
USER_QUESTION = (
    "I want to get my daughter more interested in science. What kind of gifts should I get her?"
)

question_embed = vo.embed([USER_QUESTION], model="voyage-2", input_type="query")
results = index.query(vector=question_embed.embeddings, top_k=5, include_metadata=True)
results

{'matches': [{'id': 'description_1771',
              'metadata': {'description': 'Product Name: Scientific Explorer '
                                          'My First Science Kids Science '
                                          'Experiment Kit\n'
                                          '\n'
                                          'About Product: Experiments to spark '
                                          'creativity and curiosity | Grow '
                                          'watery crystals, create a rainbow '
                                          'in a plate, explore the science of '
                                          'color and more | Represents STEM '
                                          '(Science, Technology, Engineering, '
                                          'Math) principles – open ended toys '
                                          'to construct, engineer, explorer '
                                          'and experiment | Inclu

## 검색 최적화

이 결과도 좋지만 더 최적화할 수 있습니다. Claude를 사용하여 사용자 질문에서 검색 keyword를 생성할 수 있습니다. 이를 통해 index에서 폭넓고 다양한 검색을 수행하여 더 관련성 높은 제품 설명을 얻을 수 있습니다.

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)


def get_completion(prompt):
    completion = client.completions.create(
        model="claude-2.1",
        prompt=prompt,
        max_tokens_to_sample=1024,
    )
    return completion.completion

In [ ]:
def create_keyword_prompt(question):
    return f"""\n\nHuman: Given a question, generate a list of 5 very diverse search keywords that can be used to search for products on Amazon.

The question is: {question}

Output your keywords as a JSON that has one property "keywords" that is a list of strings. Only output valid JSON.\n\nAssistant:{{"""

Anthropic client를 설정하고 prompt를 만들었으므로 이제 질문에서 keyword 생성을 시작할 수 있습니다. Claude의 output에서 쉽게 parsing할 수 있도록 keyword를 JSON object로 출력하겠습니다.

In [ ]:
keyword_json = "{" + get_completion(create_keyword_prompt(USER_QUESTION))
print(keyword_json)

In [ ]:
import json

# JSON에서 keyword 추출
data = json.loads(keyword_json)
keywords_list = data["keywords"]
print(keywords_list)

이제 keyword가 목록에 있으므로 각각 embedding하여 index에 query하고 가장 관련성 높은 상위 3개 제품 설명을 반환하겠습니다.

In [ ]:
results_list = []
for keyword in keywords_list:
    # Keyword의 embedding 가져오기
    query_embed = vo.embed([keyword], model="voyage-2", input_type="query")
    # Pinecone index에서 embedding 검색
    search_results = index.query(vector=query_embed.embeddings, top_k=3, include_metadata=True)
    # 목록에 검색 결과 추가
    for search_result in search_results.matches:
        results_list.append(search_result["metadata"]["description"])
print(len(results_list))

## Claude로 답하기

제품 설명 목록을 얻었으므로 Claude가 학습한 검색 template으로 형식을 정리하고 다른 prompt에 전달하겠습니다.

In [47]:
# 검색 결과 형식 정리
def format_results(extracted: list[str]) -> str:
    result = "\n".join(
        [
            f'<item index="{i + 1}">\n<page_content>\n{r}\n</page_content>\n</item>'
            for i, r in enumerate(extracted)
        ]
    )

    return f"\n<search_results>\n{result}\n</search_results>"


def create_answer_prompt(results_list, question):
    return f"""\n\nHuman: {format_results(results_list)} Using the search results provided within the <search_results></search_results> tags, please answer the following question <question>{question}</question>. Do not reference the search results in your answer.\n\nAssistant:"""

마지막으로 원래 사용자 질문을 묻고 Claude의 답을 받겠습니다.

In [50]:
answer = get_completion(create_answer_prompt(results_list, USER_QUESTION))
print(answer)

 To get your daughter more interested in science, I would recommend getting her an age-appropriate science kit or set that allows for hands-on exploration and experimentation. For example, for a younger child you could try a beginner chemistry set, magnet set, or crystal growing kit. For an older child, look for kits that tackle more advanced scientific principles like physics, engineering, robotics, etc. The key is choosing something that sparks her natural curiosity and lets her actively investigate concepts through activities, observations, and discovery. Supplement the kits with science books, museum visits, documentaries, and conversations about science she encounters in everyday life. Making science fun and engaging is crucial for building her interest.
